In [4]:
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
import os
from langchain_mcp_adapters.client import MultiServerMCPClient

github_pat = os.getenv("GITHUB_PAT")


#### streamable_http

In [ ]:

if not github_pat:
    raise ValueError(".env에 GITHUB_PAT를 설정해주세요.")

mcp_client = MultiServerMCPClient({
    "github": {
        "url": "https://api.githubcopilot.com/mcp/",
        "headers": {
            "Authorization": f"Bearer {github_pat}",
        },
        "transport" : "streamable_http"
    },
})

# 꼭 Authorization토큰 헤더를 Bearer 토큰으로 넣어서 요청해야 함
# 공식문서대로 하면 에러가 발생한다

In [ ]:

# Jupyter에서는 최상위 await 사용 가능
# 도구 리스트 로드
tool_list = await mcp_client.get_tools()

print("도구 개수:", len(tool_list))
print([tool.name for tool in tool_list])


#### stdio

```text

1. args 에 추가
        "-e",
        "GITHUB_TOOLSETS", 


2. "env" 환경변수에 추가
        "GITHUB_TOOLSETS": "context,pull_requests",
        

3. transport 연결 추가
        "transport": "stdio"
```

In [ ]:

mcp_client = MultiServerMCPClient({
    "github": {
        "command": "docker",
        "args": [
        "run",
        "-i",
        "--rm",
        "-e",
        "GITHUB_TOOLSETS",
        "-e",
        "GITHUB_PERSONAL_ACCESS_TOKEN",
        "-e",
        "GITHUB_HOST",
        "ghcr.io/github/github-mcp-server"
        ],
        "env": {
            "GITHUB_TOOLSETS": "context,pull_requests",
            "GITHUB_PERSONAL_ACCESS_TOKEN": github_pat
        },
        "transport": "stdio"
    }
}
)

In [ ]:
from pathlib import Path

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_mcp_adapters.tools import load_mcp_tools

# 위에서 생성한 클라이언트의 GitHub 설정 재사용
config = mcp_client.connections["github"]

server_params = StdioServerParameters(
    command=config["command"],
    args=config["args"],
    env=config.get("env"),
    cwd=config.get("cwd"),
)

log_path = Path.cwd() / "mcp_stdio_stderr.log"

# Jupyter 출력 스트림 대신 실제 파일로 오류 로그 전달
with log_path.open("w", encoding="utf-8") as errlog:
    async with stdio_client(
        server_params,
        errlog=errlog,
    ) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            tool_list = await load_mcp_tools(session)

            print("도구 개수:", len(tool_list))
            print([tool.name for tool in tool_list])

            # 이 도구를 이용한 에이전트 실행도 세션이 열려 있는 이 위치에서 수행한다.

도구 개수: 12
['add_comment_to_pending_review', 'add_reply_to_pull_request_comment', 'create_pull_request', 'get_me', 'list_pull_requests', 'merge_pull_request', 'pull_request_read', 'pull_request_review_write', 'search_pull_requests', 'ui_get', 'update_pull_request', 'update_pull_request_branch']


#### tool_list 확인

In [8]:
tool_list

[StructuredTool(name='add_comment_to_pending_review', description="Add review comment to the requester's latest pending pull request review. A pending review needs to already exist to call this (check with the user if not sure).", args_schema={'type': 'object', 'properties': {'body': {'type': 'string', 'description': 'The text of the review comment'}, 'line': {'type': 'number', 'description': 'The line of the blob in the pull request diff that the comment applies to. For multi-line comments, the last line of the range'}, 'owner': {'type': 'string', 'description': 'Repository owner', 'x-mcp-header': 'owner'}, 'path': {'type': 'string', 'description': 'The relative path to the file that necessitates a comment'}, 'pullNumber': {'type': 'number', 'description': 'Pull request number'}, 'repo': {'type': 'string', 'description': 'Repository name', 'x-mcp-header': 'repo'}, 'side': {'type': 'string', 'description': 'The side of the diff to comment on. LEFT indicates the previous state, RIGHT in

In [9]:
from langchain_ollama import ChatOllama


llm = ChatOllama(
    model='llama3.1:latest', 
    num_predict=512,
)

In [ ]:
from langchain.agents import create_agent


# 에이전트 자체에 목적이 명확하다면, 장황하게 프롬프트를 작성해서 목적을 알려줘도 되지만, 
# 단순히 도구만 가지고 작업을 하는 걸 목표로 한다면, 아래처럼 프롬프트를 간단하게 작성해도 된다
# 도구 자체에 설명이 자세하게 되어있기 때문임
agent = create_agent(
    model=llm,
    tools=tool_list,
    system_prompt = "Use the tools provided to you to answer the user's question"
)


In [ ]:
# async로 도구 리스트를 불러오기 때문에, stream 을 활용하기 위해서 별도의 헬퍼 함수를 활용

async def process_stream(stream_generator):
    results = []
    try:
        async for chunk in stream_generator:

            key = list(chunk.keys())[0]
            
            if key == 'agent':
                # Agent 메시지의 내용을 가져옴. 메세지가 비어있는 경우 어떤 도구를 어떻게 호출할지 정보를 가져옴
                content = chunk['agent']['messages'][0].content if chunk['agent']['messages'][0].content != '' else chunk['agent']['messages'][0].additional_kwargs
                print(f"'agent': '{content}'")
            
            elif key == 'tools':
                # 도구 메시지의 내용을 가져옴
                for tool_msg in chunk['tools']['messages']:
                    print(f"'tools': '{tool_msg.content}'")
            
            results.append(chunk)
        return results
    except Exception as e:
        print(f"Error processing stream: {e}")
        return results

In [ ]:
from langchain_core.messages import HumanMessage

# 개인 url
http_query = """깃헙의 Pull Request를 확인하고 코드 리뷰를 작성해주세요
PR URL: https://github.com/jasonkang14/sat-reading-client/pull/4

"""

stdio_query = """깃헙의 Pull Request를 확인하고 코드 리뷰를 작성해주세요
코드리뷰를 PR에 comment 로 남겨주세요.

PR URL: https://github.com/jasonkang14/sat-reading-client/pull/4

"""

stream_generator = agent.astream({'messages' : [HumanMessage(content=stdio_query)]})

In [ ]:
all_chunks = await process_stream(stream_generator)

if all_chunks:
    final_result = all_chunks[-1]
    print(final_result)